# OpenRSI × Frontis-MA1-35B on Colab (A100 80GB)

Runs the OpenMLE-Evo self-improvement loop over the cyber-ML tasks using a
**self-hosted Frontis-MA1-35B** served locally by vLLM — no API key.

**Requires an A100 80GB runtime** (Runtime → Change runtime type → A100). The
BF16 model is ~70 GB and fits at TP=1 on 80 GB. On a 40 GB card, use the
quantized path noted in cell 4.

Colab caveats this notebook handles for you:
- **Disk / re-download** → the HF snapshot is cached to Google Drive, so a
  session restart doesn't re-pull 70 GB.
- **Session death** → the run's output dir lives on Drive and checkpoints to
  `journal.jsonl`, so a disconnect doesn't lose progress.
- **Long sweeps** → run one task at a time with a bounded `step_limit` (cell 6)
  rather than the whole set unattended.

Run the cells top to bottom.

## 1. Confirm the GPU (must be an 80GB A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Expect: NVIDIA A100-SXM4-80GB, 81920 MiB, ...
# If you see 40960 MiB (40GB) or a T4/L4, use the quantized path in cell 4.

## 2. Mount Drive + point the HF cache and outputs at it

Everything expensive (model weights, run journals, the paper report) lands on
Drive so it survives a session restart.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

PERSIST = '/content/drive/MyDrive/openrsi_frontis'
os.makedirs(PERSIST, exist_ok=True)

# Cache the ~70GB HF snapshot on Drive (first run downloads; later runs reuse).
os.environ['HF_HOME'] = f'{PERSIST}/hf'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# Where run journals + the paper report will be kept.
OUTPUT_DIR = f'{PERSIST}/runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('HF_HOME   =', os.environ['HF_HOME'])
print('OUTPUT_DIR=', OUTPUT_DIR)

## 3. Clone the repo, check out the branch, install deps

In [ ]:
%cd /content
![ -d OpenRSI ] || git clone https://github.com/giannisp09/OpenRSI.git
%cd /content/OpenRSI
!git checkout feat/cyberml-local-model-and-paper-harness && git pull --ff-only || true
!pip install -q uv
!uv sync
!uv pip install -q "vllm>=0.6"

import subprocess
PY = subprocess.check_output(
    ['uv', 'run', 'python', '-c', 'import sys;print(sys.executable)'],
    cwd='/content/OpenRSI/OpenMLE-Evo').decode().strip()
print('candidate python =', PY)

## 4. Serve Frontis-MA1-35B with vLLM (background)

BF16, TP=1, on the 80GB A100. `--reasoning-parser qwen3` routes chain-of-thought
into a separate `reasoning_content` field so the operators' code-block parsing
still works (and the report can show a reasoning trace).

First launch downloads ~70 GB into the Drive HF cache — slow once, fast after.

**On a 40GB card instead:** add `"--quantization", "awq",` and swap the model id
for a community AWQ/GPTQ-INT4 build of Frontis-MA1 (~20 GB).

In [ ]:
import subprocess, time, urllib.request, os

log = open(f'{OUTPUT_DIR}/vllm_server.log', 'w')
srv = subprocess.Popen(
    [
        'vllm', 'serve', 'FrontisAI/Frontis-MA1-35B',
        '--served-model-name', 'frontis-ma1',
        '--max-model-len', '24576',        # bump toward 32768 if KV fits
        '--reasoning-parser', 'qwen3',
        '--enable-prefix-caching',
        '--gpu-memory-utilization', '0.95',
        '--port', '8000',
    ],
    stdout=log, stderr=subprocess.STDOUT, env={**os.environ},
)

# Wait for the endpoint (first run includes the weight download — be patient).
ready = False
for i in range(360):  # up to ~60 min
    if srv.poll() is not None:
        raise RuntimeError(f'vLLM exited early (code {srv.returncode}); see {OUTPUT_DIR}/vllm_server.log')
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=5)
        ready = True
        break
    except Exception:
        time.sleep(10)
print('server up' if ready else 'timed out — check the log')
assert ready

## 5. (optional) Sanity-check the served model

In [ ]:
import json, urllib.request
req = urllib.request.Request(
    'http://127.0.0.1:8000/v1/chat/completions',
    data=json.dumps({
        'model': 'frontis-ma1',
        'messages': [{'role': 'user', 'content': 'Reply with the single word: ready'}],
        'max_tokens': 16,
    }).encode(),
    headers={'Content-Type': 'application/json'},
)
print(json.loads(urllib.request.urlopen(req).read())['choices'][0]['message'])

## 6. Run the self-improvement loop (one task at a time)

Data is already committed in the repo, so `--skip-download` needs no network.
Set `TASK` and a bounded `step_limit` so the run finishes inside one Colab
session; journals stream to the Drive `OUTPUT_DIR` and checkpoint as they go.

Tasks: `nsl-kdd-nids`, `unsw-nb15-nids`, `phishing-url`, `dga-domains-dns`,
`clamp-pe-malware`, `tuandromd-android`. Re-run this cell per task.

In [ ]:
%cd /content/OpenRSI/OpenMLE-Evo
import os
TASK = 'nsl-kdd-nids'   # <- change per task
os.environ['PRIMARY_KEY'] = 'EMPTY'   # local endpoint needs no real key

!uv run python scripts/run_naturebench_local.py \
  --naturebench-repo .cyberml/local_naturebench \
  --local-python "{PY}" \
  --data-dir .cyberml/data --skip-download \
  --task {TASK} \
  --output-dir "{OUTPUT_DIR}/{TASK}" \
  --model-base-url http://127.0.0.1:8000/v1 --model-id frontis-ma1 \
  -- llm_concurrency=4 search.runner.solver.step_limit=25

## 7. Build the paper-ready results pack (tables, figures, report)

Writes `tables/`, `plots/` (PDF+PNG), `manifest.json`, and `REPORT.md`, then
copies the whole pack onto Drive so it outlives the session.

In [ ]:
%cd /content/OpenRSI/OpenMLE-Evo
!uv run python .cyberml/tools/make_paper_report.py \
  --output-dir "{OUTPUT_DIR}" \
  --paper-dir "{OUTPUT_DIR}/paper" \
  --model-label "Frontis-MA1-35B" \
  --all-tasks

print('\nPaper pack on Drive:', f'{OUTPUT_DIR}/paper')
!ls -R "{OUTPUT_DIR}/paper" | head -60

## 8. (optional) Stop the server / free the GPU

In [ ]:
try:
    srv.terminate(); srv.wait(timeout=30)
    print('server stopped')
except Exception as e:
    print('already stopped:', e)